# Team Rocket Visualization and Analytics Notebook

This notebook converts the visualization-heavy Python logic from the project into a Jupyter-friendly workflow.

What this version improves:
- Adds insightful comments and interpretation-oriented notes
- Beautifies charts with consistent styling, hover labels, legends, and axis titles
- Keeps charts interactive using Plotly inside Jupyter
- Organizes the analysis into sections that are easier to demo and explain


## Notebook Notes

If a chart renders in Jupyter, it is already interactive by default: zoom, pan, hover, legend toggling, and image export.


In [ ]:
from pathlib import Path
import glob
from typing import Dict, List, Optional

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pio.renderers.default = "notebook_connected"
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = px.colors.sequential.Tealgrn

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
FRONTEND_DATA_DIR = PROJECT_ROOT / "frontend" / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Frontend data dir exists: {FRONTEND_DATA_DIR.exists()}")


In [ ]:
def style_figure(fig, title: str, x_title: str = "", y_title: str = "", legend_title: str = "", height: int = 520, show_range_slider: bool = False):
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        height=height,
        hovermode="x unified",
        legend_title_text=legend_title,
        paper_bgcolor="#f7fafc",
        plot_bgcolor="#ffffff",
        margin=dict(l=40, r=20, t=70, b=40),
        font=dict(size=13, color="#1f2937"),
    )
    fig.update_xaxes(title_text=x_title, showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False)
    fig.update_yaxes(title_text=y_title, showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False)
    if show_range_slider:
        fig.update_xaxes(rangeslider_visible=True)
    return fig


def highlight_table(title: str, frame: pd.DataFrame) -> pd.DataFrame:
    print(title)
    return frame


## Shared Data Loader Functions

These come from `backend/data_loader.py`, but the Streamlit cache decorators have been removed so they work directly in Jupyter.


In [ ]:
def load_all_data() -> pd.DataFrame:
    files = glob.glob(str(DATA_DIR / "*.csv"))
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df.columns = df.columns.str.strip().str.lower()
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df


def get_available_expiries() -> List[str]:
    df = load_all_data()
    if df.empty or "expiry" not in df.columns:
        return []
    return sorted(df["expiry"].dropna().astype(str).unique())


def load_expiry_data(expiry: str) -> pd.DataFrame:
    df = load_all_data()
    if df.empty:
        return df
    if "expiry" not in df.columns:
        return pd.DataFrame()
    return df[df["expiry"].astype(str) == str(expiry)].copy()


def filter_strike_range(df: pd.DataFrame, strike_min: float, strike_max: float) -> pd.DataFrame:
    if df.empty or "strike" not in df.columns:
        return df
    return df[(df["strike"] >= strike_min) & (df["strike"] <= strike_max)].copy()


## Shared Analytics Functions

These helper functions preserve the project's backend calculations so the notebook remains faithful to the original logic.


In [ ]:
def _normalize_oi_columns(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.copy()
    normalized.columns = [col.strip() for col in normalized.columns]
    rename_map = {}
    if "oi_ce" in normalized.columns and "oi_CE" not in normalized.columns:
        rename_map["oi_ce"] = "oi_CE"
    if "oi_pe" in normalized.columns and "oi_PE" not in normalized.columns:
        rename_map["oi_pe"] = "oi_PE"
    if rename_map:
        normalized = normalized.rename(columns=rename_map)
    return normalized


def calculate_pcr(df: pd.DataFrame) -> Dict[str, float]:
    if df.empty:
        return {"call_oi": 0.0, "put_oi": 0.0, "pcr": 0.0}
    normalized = _normalize_oi_columns(df)
    call_oi = float(normalized.get("oi_CE", pd.Series(dtype=float)).fillna(0).sum())
    put_oi = float(normalized.get("oi_PE", pd.Series(dtype=float)).fillna(0).sum())
    pcr = 0.0 if call_oi == 0 else put_oi / call_oi
    return {"call_oi": call_oi, "put_oi": put_oi, "pcr": pcr}


def classify_sentiment(pcr: float) -> str:
    if pcr < 0.8:
        return "Bullish"
    if pcr > 1.2:
        return "Bearish"
    return "Neutral"


def calculate_pcr_timeseries(df: pd.DataFrame, limit: int = 50) -> List[Dict]:
    if df.empty or "datetime" not in df.columns:
        return []
    normalized = df.rename(columns={"oi_ce": "oi_CE", "oi_pe": "oi_PE"}).copy()
    normalized["datetime"] = pd.to_datetime(normalized["datetime"], errors="coerce")
    normalized = normalized.dropna(subset=["datetime"])
    if normalized.empty:
        return []
    rows: List[Dict] = []
    for timestamp, group in normalized.groupby("datetime", sort=True):
        metrics = calculate_pcr(group)
        rows.append({"datetime": timestamp, "pcr": round(metrics["pcr"], 6), "call_oi": metrics["call_oi"], "put_oi": metrics["put_oi"]})
    return rows[-limit:]


def calculate_rolling_pcr_summary(df: pd.DataFrame, window: int = 5) -> Dict:
    timeseries = calculate_pcr_timeseries(df, limit=200)
    if not timeseries:
        return {"latest_pcr": 0.0, "rolling_pcr": 0.0, "samples": 0}
    frame = pd.DataFrame(timeseries)
    frame["rolling_pcr"] = frame["pcr"].rolling(window=window, min_periods=1).mean()
    latest = frame.iloc[-1]
    return {"latest_pcr": float(latest["pcr"]), "rolling_pcr": float(latest["rolling_pcr"]), "samples": int(len(frame))}


def _normalized_df(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.rename(columns={"oi_ce": "oi_CE", "oi_pe": "oi_PE"}).copy()
    normalized["strike"] = pd.to_numeric(normalized.get("strike"), errors="coerce")
    normalized["oi_CE"] = pd.to_numeric(normalized.get("oi_CE"), errors="coerce").fillna(0)
    normalized["oi_PE"] = pd.to_numeric(normalized.get("oi_PE"), errors="coerce").fillna(0)
    return normalized.dropna(subset=["strike"])


def calculate_top_oi_strikes(df: pd.DataFrame, limit: int = 10) -> List[Dict]:
    if df.empty:
        return []
    grouped = (_normalized_df(df).assign(total_oi=lambda x: x["oi_CE"] + x["oi_PE"]).groupby("strike", as_index=False)["total_oi"].sum().sort_values("total_oi", ascending=False).head(limit))
    return [{"strike": float(r["strike"]), "total_oi": float(r["total_oi"])} for _, r in grouped.iterrows()]


def summarize_call_put_oi(df: pd.DataFrame) -> Dict[str, float]:
    normalized = _normalized_df(df)
    return {"call_oi": float(normalized["oi_CE"].sum()), "put_oi": float(normalized["oi_PE"].sum())}


In [ ]:
def _group_strike_oi(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.rename(columns={"oi_ce": "oi_CE", "oi_pe": "oi_PE"})
        .assign(
            oi_CE=lambda x: pd.to_numeric(x.get("oi_CE"), errors="coerce").fillna(0),
            oi_PE=lambda x: pd.to_numeric(x.get("oi_PE"), errors="coerce").fillna(0),
            strike=lambda x: pd.to_numeric(x.get("strike"), errors="coerce"),
        )
        .dropna(subset=["strike"])
        .groupby("strike", as_index=False)[["oi_CE", "oi_PE"]]
        .sum()
        .sort_values("strike")
    )


def calculate_max_pain(df: pd.DataFrame) -> Dict[str, Optional[float]]:
    if df.empty or "strike" not in df.columns:
        return {"max_pain_strike": None, "pain_value": None}
    grouped = _group_strike_oi(df)
    if grouped.empty:
        return {"max_pain_strike": None, "pain_value": None}
    strikes = grouped.to_dict("records")
    min_pain = float("inf")
    max_pain_strike = None
    for strike_row in strikes:
        strike = strike_row["strike"]
        pain = 0.0
        for option_row in strikes:
            option_strike = option_row["strike"]
            if strike > option_strike:
                pain += (strike - option_strike) * option_row["oi_CE"]
            elif strike < option_strike:
                pain += (option_strike - strike) * option_row["oi_PE"]
        if pain < min_pain:
            min_pain = pain
            max_pain_strike = strike
    return {"max_pain_strike": max_pain_strike, "pain_value": min_pain}


def calculate_oi_change_by_strike(df: pd.DataFrame, limit: int = 10) -> List[Dict]:
    if df.empty or "datetime" not in df.columns or "strike" not in df.columns:
        return []
    normalized = df.rename(columns={"oi_ce": "oi_CE", "oi_pe": "oi_PE"}).copy()
    normalized["datetime"] = pd.to_datetime(normalized["datetime"], errors="coerce")
    normalized["strike"] = pd.to_numeric(normalized.get("strike"), errors="coerce")
    normalized["oi_CE"] = pd.to_numeric(normalized.get("oi_CE"), errors="coerce").fillna(0)
    normalized["oi_PE"] = pd.to_numeric(normalized.get("oi_PE"), errors="coerce").fillna(0)
    normalized = normalized.dropna(subset=["datetime", "strike"])
    if normalized.empty:
        return []
    latest_points = sorted(normalized["datetime"].unique())
    if len(latest_points) < 2:
        return []
    current_time = latest_points[-1]
    previous_time = latest_points[-2]
    current = (normalized.loc[normalized["datetime"] == current_time].assign(total_oi=lambda x: x["oi_CE"] + x["oi_PE"]).groupby("strike", as_index=False)["total_oi"].sum().rename(columns={"total_oi": "current_total_oi"}))
    previous = (normalized.loc[normalized["datetime"] == previous_time].assign(total_oi=lambda x: x["oi_CE"] + x["oi_PE"]).groupby("strike", as_index=False)["total_oi"].sum().rename(columns={"total_oi": "previous_total_oi"}))
    merged = current.merge(previous, on="strike", how="outer").fillna(0)
    merged["oi_change"] = merged["current_total_oi"] - merged["previous_total_oi"]
    merged["abs_change"] = merged["oi_change"].abs()
    merged = merged.sort_values("abs_change", ascending=False).head(limit)
    return [{"strike": float(r["strike"]), "current_total_oi": float(r["current_total_oi"]), "previous_total_oi": float(r["previous_total_oi"]), "oi_change": float(r["oi_change"])} for _, r in merged.iterrows()]


def detect_unusual_oi(df: pd.DataFrame, limit: int = 5) -> List[Dict]:
    if df.empty or "strike" not in df.columns:
        return []
    normalized = df.rename(columns={"oi_ce": "oi_CE", "oi_pe": "oi_PE"}).copy()
    normalized["strike"] = pd.to_numeric(normalized.get("strike"), errors="coerce")
    normalized["oi_CE"] = pd.to_numeric(normalized.get("oi_CE"), errors="coerce").fillna(0)
    normalized["oi_PE"] = pd.to_numeric(normalized.get("oi_PE"), errors="coerce").fillna(0)
    normalized = normalized.dropna(subset=["strike"])
    normalized["total_oi"] = normalized["oi_CE"] + normalized["oi_PE"]
    grouped = (normalized.groupby("strike", as_index=False)["total_oi"].sum().sort_values("total_oi", ascending=False).head(limit))
    if grouped.empty:
        return []
    baseline = float(grouped["total_oi"].mean()) or 1.0
    return [{"strike": float(r["strike"]), "total_oi": float(r["total_oi"]), "anomaly_score": float(r["total_oi"]) / baseline} for _, r in grouped.iterrows()]


## Dataset Setup

This cell replaces the Streamlit sidebar with notebook variables.

Interpretation note:
- working with one expiry at a time mirrors how traders inspect a specific options chain
- narrowing by strike range helps remove far OTM noise when you want a cleaner signal


In [ ]:
df_all = load_all_data()
expiries = get_available_expiries()
if not expiries:
    raise ValueError(f"No expiries found in {DATA_DIR}")
EXPIRY = expiries[0]
df = load_expiry_data(EXPIRY)
df.columns = df.columns.str.strip().str.lower()
strike_min = float(df["strike"].min()) if "strike" in df.columns else None
strike_max = float(df["strike"].max()) if "strike" in df.columns else None
df = filter_strike_range(df, strike_min, strike_max) if strike_min is not None else df
print(f"Selected expiry: {EXPIRY}")
print(f"Rows in selected slice: {len(df):,}")
print(f"Unique strikes: {df['strike'].nunique():,}")
print(f"Time points: {df['datetime'].nunique():,}" if 'datetime' in df.columns else "No datetime column")
df.head()


## Market Overview

Inferential angle:
- high total put activity relative to call activity can suggest defensive positioning
- strongest OI concentrations often act like candidate support and resistance zones


In [ ]:
spot = round(df["spot_close"].iloc[-1], 2)
total_volume = int(df["volume_ce"].sum() + df["volume_pe"].sum())
total_oi = int(df["oi_ce"].sum() + df["oi_pe"].sum())
avg_pcr = round(df["volume_pe"].sum() / (df["volume_ce"].sum() + 1), 2)
highlight_table("Overview metrics", pd.DataFrame({"Metric": ["NIFTY Spot", "Total Volume", "Total Open Interest", "Volume PCR"], "Value": [spot, total_volume, total_oi, avg_pcr]}))


In [ ]:
fig = go.Figure(go.Indicator(mode="gauge+number", value=avg_pcr, title={"text": "Put-Call Ratio"}, gauge={"axis": {"range": [0, 2]}, "bar": {"color": "#0f766e"}, "steps": [{"range": [0, 0.8], "color": "#fde68a"}, {"range": [0.8, 1.2], "color": "#dbeafe"}, {"range": [1.2, 2], "color": "#fecaca"}] }))
style_figure(fig, "PCR Sentiment Gauge")
fig.show()
oi_df = df.groupby("strike")[["oi_ce", "oi_pe"]].sum().reset_index()
fig = px.bar(oi_df, x="strike", y=["oi_ce", "oi_pe"], barmode="group", labels={"value": "Open Interest", "strike": "Strike", "variable": "Option Side"}, color_discrete_sequence=["#2563eb", "#dc2626"])
fig.add_vline(x=spot, line_dash="dash", line_color="#111827", annotation_text=f"Spot {spot}")
style_figure(fig, "Open Interest Distribution by Strike", "Strike", "Open Interest", "Series")
fig.show()
vol_df = df.groupby("strike")[["volume_ce", "volume_pe"]].sum().reset_index()
fig = px.bar(vol_df, x="strike", y=["volume_ce", "volume_pe"], barmode="group", labels={"value": "Volume", "strike": "Strike", "variable": "Option Side"}, color_discrete_sequence=["#0284c7", "#ea580c"])
fig.add_vline(x=spot, line_dash="dash", line_color="#111827", annotation_text=f"Spot {spot}")
style_figure(fig, "Volume Distribution by Strike", "Strike", "Volume", "Series")
fig.show()
max_call = oi_df.loc[oi_df["oi_ce"].idxmax(), "strike"]
max_put = oi_df.loc[oi_df["oi_pe"].idxmax(), "strike"]
signal = "Neutral"
if avg_pcr > 1.2 and max_put > max_call:
    signal = "Bullish"
elif avg_pcr < 0.8 and max_call > max_put:
    signal = "Bearish"
highlight_table("Support / resistance inference", pd.DataFrame({"Resistance (Max Call OI)": [max_call], "Support (Max Put OI)": [max_put], "Market Signal": [signal]}))


## Price Analysis

Interpretation note:
- the spot chart shows the underlying path
- the ATM option chart tracks the premium closest to the current market center
- rolling volatility is not true implied volatility, but it is still a useful stress proxy


In [ ]:
required_cols = ["datetime", "spot_close", "strike", "ce", "pe"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
fig = px.line(df.sort_values("datetime"), x="datetime", y="spot_close", markers=True, labels={"datetime": "Time", "spot_close": "Spot Close"})
style_figure(fig, "NIFTY Spot Price Over Time", "Time", "Spot Close", show_range_slider=True)
fig.show()
temp = df.copy().sort_values("datetime")
temp["distance_from_spot"] = (temp["strike"] - temp["spot_close"]).abs()
atm_df = temp.loc[temp.groupby("datetime")["distance_from_spot"].idxmin()].sort_values("datetime")
fig = go.Figure()
fig.add_trace(go.Scatter(x=atm_df["datetime"], y=atm_df["ce"], name="ATM Call Price", mode="lines+markers", line=dict(color="#2563eb")))
fig.add_trace(go.Scatter(x=atm_df["datetime"], y=atm_df["pe"], name="ATM Put Price", mode="lines+markers", line=dict(color="#dc2626")))
style_figure(fig, "ATM Option Prices Over Time", "Time", "Premium", "Contract", show_range_slider=True)
fig.show()
strike_df = df.groupby("strike")[["ce", "pe"]].mean().reset_index()
fig = px.line(strike_df, x="strike", y=["ce", "pe"], markers=True, labels={"value": "Average Premium", "strike": "Strike", "variable": "Series"}, color_discrete_sequence=["#1d4ed8", "#b91c1c"])
style_figure(fig, "Average Option Prices by Strike", "Strike", "Average Premium", "Series")
fig.show()
fig = px.histogram(df, x="spot_close", nbins=40, marginal="box", labels={"spot_close": "Spot Close"}, color_discrete_sequence=["#0f766e"])
style_figure(fig, "Spot Price Distribution", "Spot Close", "Frequency")
fig.show()
temp["returns"] = temp["spot_close"].pct_change()
temp["volatility"] = temp["returns"].rolling(20).std() * (252 ** 0.5)
fig = px.line(temp, x="datetime", y="volatility", labels={"datetime": "Time", "volatility": "Annualized Rolling Volatility"}, color_discrete_sequence=["#7c3aed"])
style_figure(fig, "Rolling Volatility Proxy (20 Period)", "Time", "Annualized Volatility", show_range_slider=True)
fig.show()
df.head(20)


## Open Interest Analysis

Interpretation note:
- a positive `put - call` OI gap can imply stronger put-side positioning at that strike
- heatmaps help identify where positioning is persistent across time, not just large in aggregate


In [ ]:
oi_group = df.groupby("strike")[["oi_ce", "oi_pe"]].sum().reset_index()
fig = px.bar(oi_group, x="strike", y=["oi_ce", "oi_pe"], barmode="group", labels={"value": "Open Interest", "strike": "Strike", "variable": "Series"}, color_discrete_sequence=["#2563eb", "#dc2626"])
style_figure(fig, "Call vs Put Open Interest by Strike", "Strike", "Open Interest", "Series")
fig.show()
oi_group["oi_diff"] = oi_group["oi_pe"] - oi_group["oi_ce"]
fig = px.bar(oi_group, x="strike", y="oi_diff", color="oi_diff", color_continuous_scale="RdBu", labels={"strike": "Strike", "oi_diff": "Put OI - Call OI"})
style_figure(fig, "Market Bias by Strike (Put OI Minus Call OI)", "Strike", "OI Bias")
fig.show()
max_call = oi_group.loc[oi_group["oi_ce"].idxmax(), "strike"]
max_put = oi_group.loc[oi_group["oi_pe"].idxmax(), "strike"]
highlight_table("Key OI levels", pd.DataFrame({"Resistance (Max Call OI)": [max_call], "Support (Max Put OI)": [max_put]}))


In [ ]:
if "datetime" in df.columns:
    heatmap_df = df.pivot_table(values="oi_ce", index="strike", columns="datetime", aggfunc="sum")
    fig = px.imshow(heatmap_df, aspect="auto", color_continuous_scale="YlGnBu", labels={"x": "Time", "y": "Strike", "color": "Call OI"})
    style_figure(fig, "Call Open Interest Heatmap", "Time", "Strike")
    fig.show()
df.head(20)


## PCR Sentiment

Interpretation note:
- `PCR > 1` means put activity outweighs call activity
- rising PCR can reflect caution, hedging, or bearish conviction depending on context


In [ ]:
temp = df.copy().sort_values("datetime")
temp["pcr_volume"] = temp["volume_pe"] / (temp["volume_ce"] + 1)
temp["pcr_oi"] = temp["oi_pe"] / (temp["oi_ce"] + 1)
pcr = temp.groupby("datetime")[["pcr_volume", "pcr_oi"]].mean().reset_index()
latest_pcr = pcr["pcr_volume"].iloc[-1]
sentiment = classify_sentiment(latest_pcr)
highlight_table("PCR snapshot", pd.DataFrame({"Latest Volume PCR": [round(latest_pcr, 3)], "Interpretation": [sentiment]}))


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pcr["datetime"], y=pcr["pcr_volume"], name="Volume PCR", mode="lines+markers", line=dict(color="#0f766e")))
fig.add_trace(go.Scatter(x=pcr["datetime"], y=pcr["pcr_oi"], name="OI PCR", mode="lines+markers", line=dict(color="#7c3aed")))
fig.add_hline(y=0.8, line_dash="dot", line_color="#16a34a", annotation_text="Bullish zone")
fig.add_hline(y=1.0, line_dash="dash", line_color="#f59e0b", annotation_text="Parity")
fig.add_hline(y=1.2, line_dash="dot", line_color="#dc2626", annotation_text="Bearish zone")
style_figure(fig, "PCR Trend Through Time", "Time", "PCR", "Series", show_range_slider=True)
fig.show()
fig = px.histogram(pcr, x="pcr_volume", nbins=40, marginal="rug", labels={"pcr_volume": "Volume PCR"}, color_discrete_sequence=["#0f766e"])
style_figure(fig, "Distribution of Volume PCR", "Volume PCR", "Frequency")
fig.show()
insight = "PCR indicates neutral sentiment"
if latest_pcr > 1.2:
    insight = "High PCR suggests strong put-side activity, which can indicate defensive or bearish positioning."
elif latest_pcr < 0.8:
    insight = "Low PCR suggests stronger call participation, which often aligns with bullish appetite."
print(insight)


## Volume Heatmap

Interpretation note:
- concentrated volume near a narrow strike band suggests attention is clustering there
- spike detection highlights unusually active contracts that may deserve closer trade-flow review


In [ ]:
temp = df.copy()
temp.columns = temp.columns.str.strip().str.lower()
temp["total_volume"] = temp["volume_ce"].fillna(0) + temp["volume_pe"].fillna(0)
pivot = temp.pivot_table(values="total_volume", index="strike", columns="expiry", aggfunc="sum")
fig = px.imshow(pivot, color_continuous_scale="Sunsetdark", aspect="auto", labels={"x": "Expiry", "y": "Strike", "color": "Total Volume"})
style_figure(fig, "Volume Heatmap (Strike vs Expiry)", "Expiry", "Strike")
fig.show()
time_pivot = temp.pivot_table(values="total_volume", index="strike", columns="datetime", aggfunc="sum")
fig = px.imshow(time_pivot, color_continuous_scale="Viridis", aspect="auto", labels={"x": "Time", "y": "Strike", "color": "Total Volume"})
style_figure(fig, "Volume Heatmap (Strike vs Time)", "Time", "Strike")
fig.show()
volume_group = temp.groupby("strike")[["volume_ce", "volume_pe"]].sum().reset_index()
fig = px.bar(volume_group, x="strike", y=["volume_ce", "volume_pe"], barmode="group", labels={"value": "Volume", "strike": "Strike", "variable": "Series"}, color_discrete_sequence=["#0284c7", "#ea580c"])
style_figure(fig, "Call vs Put Volume by Strike", "Strike", "Volume", "Series")
fig.show()
threshold = temp["total_volume"].quantile(0.98)
spikes = temp[temp["total_volume"] > threshold].sort_values("total_volume", ascending=False)
highlight_table("Detected volume spikes", spikes.head(20) if not spikes.empty else pd.DataFrame({"message": ["No significant spikes detected"]}))


## Volatility Smile

Interpretation note:
- in this project, CE premium is used as a proxy rather than a true implied-volatility calculation
- the curve shape still helps show how premium changes as strikes move away from the money


In [ ]:
smile_df = df.copy()
smile_df.columns = smile_df.columns.str.strip().str.lower()
if "ce" not in smile_df.columns or "strike" not in smile_df.columns:
    raise ValueError("Missing required columns: strike, ce")
iv = smile_df.groupby("strike")["ce"].mean().reset_index()
fig = px.line(iv, x="strike", y="ce", markers=True, labels={"strike": "Strike", "ce": "Average CE Premium"}, color_discrete_sequence=["#7c3aed"])
style_figure(fig, "Volatility Smile Proxy", "Strike", "Average CE Premium")
fig.show()


## Volatility Surface

Interpretation note:
- this is best read as a premium surface, not a Black-Scholes implied-volatility surface
- it is still useful for spotting where option pricing is elevated across strike-expiry combinations


In [ ]:
surface_df = load_all_data()
surface_df.columns = surface_df.columns.str.strip().str.lower()
for col in ["strike", "expiry", "ce"]:
    if col not in surface_df.columns:
        raise ValueError(f"Missing required column: {col}")
pivot = surface_df.pivot_table(values="ce", index="strike", columns="expiry", aggfunc="mean")
if pivot.empty:
    raise ValueError("Pivot table is empty")
fig = px.imshow(pivot, aspect="auto", color_continuous_scale="Tealgrn", labels={"x": "Expiry", "y": "Strike", "color": "Average CE Premium"})
style_figure(fig, "Volatility Surface Proxy (Strike vs Expiry)", "Expiry", "Strike")
fig.show()
fig = px.histogram(surface_df, x="ce", nbins=40, marginal="box", labels={"ce": "CE Premium"}, color_discrete_sequence=["#0f766e"])
style_figure(fig, "Distribution of CE Prices", "CE Premium", "Frequency")
fig.show()


## Market Activity Clustering

Interpretation note:
- clustering is a fast unsupervised way to separate quiet strikes from unusually active ones
- the model does not know market meaning; we add interpretation after seeing the grouped behavior


In [ ]:
cluster_files = list(FRONTEND_DATA_DIR.glob("*.csv"))
cluster_df = pd.concat([pd.read_csv(file) for file in cluster_files], ignore_index=True) if cluster_files else pd.DataFrame()
if cluster_df.empty:
    raise ValueError(f"No CSV files found in {FRONTEND_DATA_DIR}")
cluster_df.columns = cluster_df.columns.str.strip().str.lower()
required_cols = ["volume_ce", "volume_pe", "oi_ce", "oi_pe", "strike"]
missing = [c for c in required_cols if c not in cluster_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
cluster_df["total_volume"] = pd.to_numeric(cluster_df["volume_ce"], errors="coerce") + pd.to_numeric(cluster_df["volume_pe"], errors="coerce")
cluster_df["total_oi"] = pd.to_numeric(cluster_df["oi_ce"], errors="coerce") + pd.to_numeric(cluster_df["oi_pe"], errors="coerce")
cluster_df = cluster_df.dropna(subset=["total_volume", "total_oi", "strike"]).copy()
n_clusters = 3
features = cluster_df[["total_volume", "total_oi"]]
if len(cluster_df) < n_clusters:
    raise ValueError("Dataset too small for clustering")
model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_df["cluster"] = model.fit_predict(features)
cluster_labels = {0: "Low Activity", 1: "Medium Activity", 2: "High Activity", 3: "Institutional", 4: "Extreme", 5: "Outlier"}
cluster_df["cluster_label"] = cluster_df["cluster"].map(cluster_labels)
highlight_table("Cluster summary metrics", pd.DataFrame({"Metric": ["Total Records", "Average Volume", "Average OI"], "Value": [len(cluster_df), int(cluster_df["total_volume"].mean()), int(cluster_df["total_oi"].mean())]}))


In [ ]:
fig = px.scatter(cluster_df, x="strike", y="total_volume", size="total_oi", color="cluster_label", hover_data=["total_oi"], labels={"strike": "Strike", "total_volume": "Total Volume", "cluster_label": "Cluster"}, color_discrete_sequence=["#93c5fd", "#60a5fa", "#1d4ed8", "#7c3aed", "#f97316", "#dc2626"])
style_figure(fig, "Options Market Activity Clusters", "Strike", "Total Volume", "Cluster")
fig.show()
cluster_summary = cluster_df.groupby("cluster_label")[["total_volume", "total_oi"]].mean().round(0).reset_index()
highlight_table("Average feature values by cluster", cluster_summary)
high_activity = cluster_df[cluster_df["cluster_label"] == "High Activity"]
if len(high_activity) > 10:
    print("Inference: heavy activity appears across multiple strikes, which may indicate broad speculative or hedging participation.")
elif len(high_activity) > 0:
    print("Inference: a smaller set of strikes is attracting unusually strong participation.")
else:
    print("Inference: activity appears comparatively balanced without a standout cluster.")


## AI Anomaly Detection

Interpretation note:
- Isolation Forest flags observations that look structurally unusual relative to the rest of the dataset
- anomalies are prompts for review, not automatic evidence of manipulation or directional certainty


In [ ]:
anomaly_df = load_all_data()
anomaly_df.columns = anomaly_df.columns.str.strip().str.lower()
required_cols = ["volume_ce", "volume_pe", "oi_ce", "oi_pe", "strike", "datetime"]
missing = [c for c in required_cols if c not in anomaly_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
anomaly_df["total_volume"] = anomaly_df["volume_ce"] + anomaly_df["volume_pe"]
anomaly_df["total_oi"] = anomaly_df["oi_ce"] + anomaly_df["oi_pe"]
contamination = 0.02
features = anomaly_df[["total_volume", "total_oi"]].fillna(0)
model = IsolationForest(contamination=contamination, random_state=42)
anomaly_df["anomaly"] = model.fit_predict(features)
anomaly_df["anomaly_label"] = anomaly_df["anomaly"].map({1: "Normal", -1: "Anomaly"})
highlight_table("Anomaly detection summary", pd.DataFrame({"Total Data Points": [len(anomaly_df)], "Detected Anomalies": [int((anomaly_df["anomaly"] == -1).sum())], "Contamination Setting": [contamination]}))


In [ ]:
fig = px.scatter(anomaly_df, x="strike", y="total_volume", size="total_oi", color="anomaly_label", hover_data=["datetime", "total_volume", "total_oi"], labels={"strike": "Strike", "total_volume": "Total Volume", "anomaly_label": "Classification"}, color_discrete_map={"Normal": "#0f766e", "Anomaly": "#dc2626"})
style_figure(fig, "AI Detected Unusual Options Activity", "Strike", "Total Volume", "Classification")
fig.show()
anomalies = anomaly_df.loc[anomaly_df["anomaly"] == -1, ["datetime", "strike", "total_volume", "total_oi"]].sort_values("total_volume", ascending=False)
highlight_table("Top detected anomalies", anomalies.head(25))


## Backend Calculation Summaries

This final block exposes the backend analytics as compact outputs that are easy to verify during a demo.


In [ ]:
calculation_summary = {
    "pcr": calculate_pcr(df),
    "rolling_pcr_summary": calculate_rolling_pcr_summary(df),
    "top_oi_strikes": calculate_top_oi_strikes(df, limit=10),
    "call_put_oi_summary": summarize_call_put_oi(df),
    "max_pain": calculate_max_pain(df),
    "oi_change_by_strike": calculate_oi_change_by_strike(df, limit=10),
    "unusual_oi": detect_unusual_oi(df, limit=5),
}
calculation_summary
